<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = (df['qty'] * df['price'])
rows = len(df)
revenue = df['revenue'].sum()
print(f'TOTAL REVENUE: {revenue}     TOTAL UNITS: {rows}')

TOTAL REVENUE: 8520.0     TOTAL UNITS: 400


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
summary = df.groupby('category', as_index=False)['revenue'].sum()
print(f'''Resultant Revenue by Category:

{summary}''')

Resultant Revenue by Category:

   category  revenue
0     Drink   1554.0
1      Food   4293.0
2     Merch   1771.5
3  RainGear    901.5


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [10]:
vendor_price = df.groupby('vendor_id')['price'].sum()
vendor_count = df.groupby('vendor_id')['price'].count()
vendor_avg = vendor_price / df.groupby('vendor_id')['qty'].sum()
print(f'Vendor Average Revenue: {vendor_avg}   and order count  {vendor_count}')

Vendor Average Revenue: vendor_id
V-01    5.696809
V-05    5.772472
V-10    5.565000
V-18    5.488479
dtype: float64   and order count  vendor_id
V-01     94
V-05     93
V-10    105
V-18    108
Name: price, dtype: int64


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
percent_revenue = merch_revenue/revenue * 100
print(f'Percentage of revenue from merch: {percent_revenue:.2f}%')

Percentage of revenue from merch: 20.79%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
orders = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
orders_revenue = orders['revenue'].sum()

print(f'before merge: {rows} rows, ${revenue:.2f}')
print(f'after merge: {len(orders)} rows, ${orders_revenue:.2f}')
print(f'Unknown vendor revenue: ${df[df['vendor_id'] == 'V-18']['revenue'].sum()}')

before merge: 400 rows, $8520.00
after merge: 400 rows, $8520.00
Unknown vendor revenue: $2349.0


The unmatched vendor is V-18. Since the revenue from this unknown is substantial, I have decided to keep it in and just say the Vendor is not known.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_df = pd.pivot_table(
    orders,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum'
)
pivot_df

category,Drink,Food,Merch,RainGear
vendor_name,,,,
Cav Merch North,502.5,1054.5,400.5,175.5
Hoos Burgers,171.0,1338.0,373.5,241.5
Rotunda Tacos,298.5,882.0,489.0,244.5


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(summary['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(orders) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. Rain Gear had significantly the lowest revenue. I would need some more numbers on cost, but I would recommend switching as a vendor from rain gear (which only brought in \$901.50) to more food (which brought in \$4293.00). Vendors should more their resources around to accomodate the more revenue that comes from food.

b. I think highest average revenue is the most deceiving. It doesn't really show the breakdown of each category. Some items are completely dependent on weather (such as rain gear) so the average revenue would not be consistent for every game. The problem is the average does not show the category breakdown. Also, one of the vendors is unmatched, which the numbers do not show.